#Environment

In [ ]:
%pip install pandas
%pip install matplotlib
%pip install tkinter
%pip install statsmodels
%pip install pmdarima

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pmdarima as pm

from statsmodels.tsa.stattools import adfuller                 
from statsmodels.tsa.seasonal import seasonal_decompose         
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf   

import statsmodels.formula.api as smf   

#1. Data Analysis


In [ ]:
data_data = pd.read_excel(
    "D:/new_pc/engineershit/Data Sets/Business Forecast Exam/exam_data.xlsx",
    sheet_name="Data"
)

data_forecast = pd.read_excel(
    "D:/new_pc/engineershit/Data Sets/Business Forecast Exam/exam_data.xlsx",
    sheet_name="Weather forecasts"
)

data_legend = pd.read_excel(
    "D:/new_pc/engineershit/Data Sets/Business Forecast Exam/exam_data.xlsx",
    sheet_name="Legend"
)

In [ ]:
#0.
data_data = pd.read_excel(
    "D:/new_pc/engineershit/Data Sets/Business Forecast Exam/exam_data.xlsx",
    sheet_name="Data"
)

data_forecast = pd.read_excel(
    "D:/new_pc/engineershit/Data Sets/Business Forecast Exam/exam_data.xlsx",
    sheet_name="Weather forecasts"
)

data_legend = pd.read_excel(
    "D:/new_pc/engineershit/Data Sets/Business Forecast Exam/exam_data.xlsx",
    sheet_name="Legend"
)

#1.
display(data_legend.shape[0]) 
display(data_legend.shape[1]) 
display(data_data.shape)
display(data_forecast.shape)

#2.
display(data_data.head(5))
display(data_forecast.head(3))
display(data_legend.head(27))

#3.
data_data.columns

#4.
data_data.dtypes

#5.
data_data["date"] = pd.to_datetime(data_data["date"])
data_forecast["date"] = pd.to_datetime(data_forecast["date"])

#6.
print(data_data["date"].dtypes)
print(data_forecast["date"].dtypes)

#7.
data_data.isnull().sum()

#8.
data_data.describe()

#9.
revenue = data_data["tr"]
revenue.describe()

#10.
rainy_days = data[data["prcp"] > 0]
dry_days = data[data["prcp"] == 0]

print(f"Number of rainy days: {rainy_days.shape[0]}")
print(f"Number of dry days: {dry_days.shape[0]}")

#11.
data_data["total_purchases"] = data_data["card"] + data_data["cash"]

display(data_data[
    ["total_purchases", "card", "cash"]
    ].head(5))

print(f"Average purchase a day: {data_data['total_purchases'].mean()}")

#12.1
data_data["total_cups"] = (
    data_data["q_1"] 
    + data_data["q_2"] 
    + data_data["q_3"] 
    + data_data["q_4"] 
    + data_data["q_5"] 
    + data_data["q_6"] 
    + data_data["q_7"] 
    + data_data["q_8"]
)

display(data_data[
    ["total_cups", "q_1", "q_2"]
    ].head(5))

#12.2
data_data["purchase_match"] = (
    data_data["total_purchases"] == data_data["total_cups"]
)


display(
    data_data["purchase_match"].value_counts()
)

#13.
correlation_data = data_data[
    ["tr", "tavg", "prcp", "wspd", "pres"]
]

correlation_matrix = correlation_data.corr()

correlation_matrix["tr"]

#14.
data_data.plot.line(
    x = "date",
    y = "tr",
    title = "Daily Revenue Over Time",
)

#15.
def summarize_data(df, column):
    summary = {
        "mean": df[column].mean(),
        "median": df[column].median(),
        "std_dev": df[column].std(),
        "min": df[column].min(),
        "max": df[column].max()
    }
    return summary

summarize_data(data_data, "tr")

#2. Forecasting

In [ ]:
#0.

data_sales = pd.read_excel(
    "D:/new_pc/engineershit/Data Sets/Business Forecast Exam/exam_data.xlsx",
    sheet_name="Data"
)

data_forecast = pd.read_excel(
    "D:/new_pc/engineershit/Data Sets/Business Forecast Exam/exam_data.xlsx",
    sheet_name="Weather forecasts"
)

data_legend = pd.read_excel(
    "D:/new_pc/engineershit/Data Sets/Business Forecast Exam/exam_data.xlsx",
    sheet_name="Legend"
)




#1. Dynammic properties

y = data_sales["tr"]

plot = y.plot(
    title = "Daily Revenue Over Time",
    xlabel = "Date",
    ylabel = "Revenue"
)

#I.
adf_result = adfuller(y.dropna())
adf_pvalue = adf_result[1]
adf_pvalue

#II.
plot_acf(y,
         lags = 30,
         title = "Autocorrelation Function (ACF) for Daily Revenue"
);

#III. 
dy = y.diff().dropna()
display(y.describe().round())
display(dy.describe().round())

plot_acf(dy,
         lags = 30,
         title = "Autocorrelation Function (ACF) for Differenced Daily Revenue"
);

y_decomp = seasonal_decompose(y,
                              model = "multiplicative",
                              period = 7)
plot = y_decomp.plot()

dy_decomp = seasonal_decompose(dy,
                               model = "additive",
                               period = 7)
plot = dy_decomp.plot()




#2. Choose independent variables

display(data_legend.describe())
display(data_legend.head(27))

x1 = data_sales["tavg"]
x2 = data_sales["tmax"]
x3 = data_sales["tmin"]
x4 = data_sales["prcp"]

adf_result_x1 = adfuller(x1)
adf_result_x2 = adfuller(x2)
adf_result_x3 = adfuller(x3)
adf_result_x4 = adfuller(x4)

print(f"ADF p-value for x1 (tavg): {adf_result_x1[1]}")
print(f"ADF p-value for x2 (tmax): {adf_result_x2[1]}")
print(f"ADF p-value for x3 (tmin): {adf_result_x3[1]}")
print(f"ADF p-value for x4 (prcp): {adf_result_x4[1]}")

dx1 = x1.diff().dropna()
dx2 = x2.diff().dropna()
dx3 = x3.diff().dropna()
dx4 = x4.diff().dropna()




#3. Regression analysis on dx1,dx2,dx3,dx4

data_sales_same_length = pd.concat(
    [dy, dx1],
    axis = 1
)

data_sales_c = data_sales_same_length.dropna()


#Regression analysis on dx1

regression_model_1 = smf.ols(
    formula = "dy ~ dx1",
    data = data_sales_c
).fit()



#Regression analysis on dx2,dx3

regression_model_2 = smf.ols(
    formula = "dy ~ dx2 + dx3",
    data = data_sales_c
).fit()


print("MODEL 1")
print(regression_model_1.summary())
print("MODEL 2")
print(regression_model_2.summary())

#Check relationships visually

#dx1 
data_sales_c.plot.scatter(
    x="tavg",
    y="tr"
)

#dx3

data_sales.plot.scatter(
 x = "tmin",
 y = "tr"   
)

#dx4

regression_model_3 = smf.ols(
    formula = "dy ~ dx4",
    data = data_sales_c
).fit()

print("MODEL 3")
print(regression_model_3.summary())

#Plot it

data_sales.plot.scatter(
    x = "prcp",
    y = "tr"
)




#4. Regresssion estimation, residuals, test

data_sales_c = pd.DataFrame({
    "dy": dy,
    "dx1": dx1
}).dropna()

#Split oos = 49

train_data = data_sales_c.iloc[:244]
test_data = data_sales_c.iloc[244:] #integer location

print(train_data.shape)
print(test_data.shape)

#Run regression on training data

#not actually estimated the regression yet. 
# - “I want an OLS regression where dy is explained by dx1, using train_data.”

regression_model_train = smf.ols(
    formula = "dy ~ dx1",
    data = train_data
)

#Estimate residuals
residuals_train = regression_model_train.fit().resid

plot_acf(residuals_train, lags=30);

#Fit 
fitted_model = regression_model_train.fit()


##Calculate residuals for test data
test_predictions = fitted_model.predict(test_data)

test_errors = test_data["dy"] - test_predictions

display(test_errors.describe().round()) 

# Performance
rmse = np.sqrt(np.mean(test_errors ** 2))
mae = np.mean(np.abs(test_errors))

print("Test RMSE:", rmse)
print("Test MAE:", mae) 


##Actual vs Predicted
performance = pd.DataFrame({
    "actual": test_data["dy"],
    "predicted": test_predictions,
    "error": test_errors
})

performance.head(10)




#5. Forecast with regression

#Final regression
final_regression = smf.ols(
    formula="dy ~ dx1",
    data=data_sales_c
).fit()

 #New data for forecasting
future["dx1"] = future["tavg"].diff()

#Fix data
future.loc[future.index[0], "dx1"] = (
    future["tavg"].iloc[0] - data_sales["tavg"].iloc[-1]
)

#Predict
future["predicted_dy"] = final_regression.predict(future)

#Convert predicted_dy to forecast_tr
last_revenue = data_sales["tr"].iloc[-1]

future["forecast_tr"] = (
    last_revenue + future["predicted_dy"].cumsum()
)

#Show forecast
future[["date", "tavg", "dx1", "predicted_dy", "forecast_tr"]]




#6. ARIMA model

y = data_sales["tr"]

train_data_arima = y.iloc[:244]
test_data_arima = y.iloc[245:]

print("Train data shape:", train_data_arima.shape)
print("Test data shape:", test_data_arima.shape)

#Train ARIMA

arima_model = pm.auto_arima(
    train_data_arima,
    seasonal=True,          #check for seasonality
    m=7,                    #weekly seasonality
    stepwise=True,          #find best model quickly
    suppress_warnings=True,  #don't show warnings
    trace=True
)

print(arima_model.summary())




#7. Forecast with ARIMA

final_arima_model = pm.ARIMA(                   #best already found
    order=arima_model.order,
    seasonal_order=arima_model.seasonal_order
)

final_arima_model.fit(y)

# Number of future days
n_future = len(data_forecast)

# Forecast + 95% confidence interval
arima_forecast, conf_int = final_arima_model.predict(
    n_periods=n_future,
    return_conf_int=True,
    alpha=0.05
)

arima_future = pd.DataFrame(
    {
    "date": data_forecast["date"].to_numpy(),
    "arima_forecast": np.asarray(arima_forecast),
    "lower_95": conf_int[:, 0],
    "upper_95": conf_int[:, 1]
}
)

arima_future




#8. Forecast combinations

#Equal-weight combination 50-50
equal_forecast = (
    0.5 * regression_test_forecast.to_numpy()
    + 0.5 * np.asarray(arima_test_forecast)
)

#Granger–Ramanathan combination
gr_model = smf.ols(
    formula="actual ~ regression + arima",
    data=combination_data
).fit()

print(gr_model.summary())